# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The queue comes from the week-5 model, scored the honest way: every page's expected CTR is an **out-of-fold** prediction from a model that never saw that page's client. The ranking score is the click gap, `max(expected_ctr - ctr, 0) / 100 * impressions_90d`.

Each page gets an **archetype** (a plain rule over its own numbers, not a learned cluster), a **reason code** a reviewer can check by eye, and an **action** mapped from the archetype:

| archetype | rule | reason code | action |
|---|---|---|---|
| visible, ignored | 0 clicks in 90d | `zero_clicks_visible` | `check_indexing_and_intent` |
| high-volume | over 50,000 impressions | `very_high_impressions` | `human_review_only` |
| under-clicking | CTR under half the expected CTR | `ctr_below_expected` | `review_title_meta` |
| holding | CTR between half and 1x expected | none | `monitor` |
| over-performing | CTR at or above expected | none | `protect` |

The rules apply top to bottom, so a page lands in the first row it matches. "High-volume" comes before "under-clicking" on purpose. A big page with a weak-looking CTR is the riskiest one to edit, and large pages often rank for branded searches (I cannot see queries here, so that is a reason for a person to look, not a finding).

In [1]:
# Setup: works in Colab (clones my repo) and locally (walks up to the repo root)
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/nadim7009/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import json, pathlib
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
OUT = pathlib.Path("work/outputs"); OUT.mkdir(parents=True, exist_ok=True)
FIG = pathlib.Path("work/figures"); FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# Same candidate pool as weeks 1 to 4: real position data, first two result pages, >= 500 impressions
pool = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)].reset_index(drop=True)
print(f"{len(pool):,} pages from {pool['client_id'].nunique()} clients")

FEATS = ["avg_position", "impressions_90d", "content_type", "main_intent", "word_count", "content_age_days"]
Y = pool["ctr"].values
W = pool["impressions_90d"].values
GROUPS = pool["client_id"]

def make_X(cols):
    X = pool[cols].copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = X[c].fillna("unknown").astype("category")
    return X

def new_model():
    return HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05,
                                         categorical_features="from_dtype", random_state=42)

def wmae(y, p, w):
    return float(np.average(np.abs(y - p), weights=w))

def evaluate(cols, splitter, groups=None):
    """Return per-fold rule error, per-fold model error, and out-of-fold predictions for both."""
    X = make_X(cols)
    rule_err, model_err = [], []
    oof_rule = np.zeros(len(pool)); oof_model = np.zeros(len(pool))
    for tr, te in splitter.split(X, Y, groups):
        tier_median = pool.iloc[tr].groupby("position_tier")["ctr"].median()
        rp = pool.iloc[te]["position_tier"].map(tier_median).fillna(pool.iloc[tr]["ctr"].median()).values
        m = new_model().fit(X.iloc[tr], Y[tr], sample_weight=W[tr])
        mp = m.predict(X.iloc[te])
        oof_rule[te], oof_model[te] = rp, mp
        rule_err.append(wmae(Y[te], rp, W[te])); model_err.append(wmae(Y[te], mp, W[te]))
    return np.array(rule_err), np.array(model_err), oof_rule, oof_model

12,023 pages from 28 clients


In [2]:
pool["client_id"] = pool["client_id"].astype(str)
_, _, _, oof_model = evaluate(FEATS, GroupKFold(n_splits=5), GROUPS)
pool["expected_ctr"] = oof_model.round(3)
pool["click_gap"] = (np.clip(pool["expected_ctr"] - pool["ctr"], 0, None) / 100 * pool["impressions_90d"]).round(1)

def archetype(r):
    if r["clicks_90d"] == 0:                      return ("visible_ignored", "zero_clicks_visible", "check_indexing_and_intent")
    if r["impressions_90d"] > 50000:              return ("high_volume", "very_high_impressions", "human_review_only")
    if r["ctr"] < 0.5 * r["expected_ctr"]:        return ("under_clicking", "ctr_below_expected", "review_title_meta")
    if r["ctr"] < r["expected_ctr"]:              return ("holding", "", "monitor")
    return ("over_performing", "", "protect")

pool[["archetype", "reason_code", "action"]] = pool.apply(archetype, axis=1, result_type="expand")
queue = pool.sort_values("click_gap", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

arch = (queue.groupby("archetype").agg(pages=("rank", "size"), impressions=("impressions_90d", "sum"),
                                       click_gap=("click_gap", "sum")).sort_values("pages", ascending=False))
arch["share_of_pages"] = (arch["pages"] / len(queue)).round(3)
print(arch.round({"impressions": 0, "click_gap": 0, "share_of_pages": 3}).to_string())

top50 = queue.head(50)
print("\nTop 50 by raw click gap, archetype mix:"); print(top50["archetype"].value_counts().to_string())
ACTIONABLE = ["review_title_meta", "check_indexing_and_intent"]
actionable = queue[queue["action"].isin(ACTIONABLE)].reset_index(drop=True)
actionable.insert(0, "action_rank", actionable.index + 1)
print(f"\nActionable pages (editor queue): {len(actionable):,}; routed to a named owner (high_volume): {int((queue['archetype'] == 'high_volume').sum())}")
print("\nTop 10 of the editor queue:"); print(actionable.head(10)[["action_rank", "position_tier", "avg_position", "impressions_90d", "clicks_90d", "ctr", "expected_ctr", "click_gap", "archetype", "reason_code", "action"]].round(2).to_string(index=False))
cols = ["rank", "position_tier", "avg_position", "impressions_90d", "clicks_90d", "ctr", "expected_ctr",
        "click_gap", "archetype", "reason_code", "action"]
print("\nTop 10:"); print(queue.head(10)[cols].round(2).to_string(index=False))

                 pages  impressions  click_gap  share_of_pages
archetype                                                     
over_performing   5011     32861359        0.0           0.417
holding           2919     22436961    20164.0           0.243
under_clicking    2464     19841880    47319.0           0.205
visible_ignored   1215      1867848     4217.0           0.101
high_volume        414     41784302    46752.0           0.034

Top 50 by raw click gap, archetype mix:
archetype
high_volume        49
visible_ignored     1

Actionable pages (editor queue): 3,679; routed to a named owner (high_volume): 414

Top 10 of the editor queue:
 action_rank position_tier  avg_position  impressions_90d  clicks_90d  ctr  expected_ctr  click_gap       archetype         reason_code                    action
           1        page_1           9.7           208678           0 0.00          0.20      421.5 visible_ignored zero_clicks_visible check_indexing_and_intent
           2        page_1 

**Reading the queue.** 49 of the top 50 by raw click gap are `high_volume` pages. The gap multiplies by impressions, so the biggest pages float up. Those are also the pages where a title change carries the most risk, so they go to a named person, not to an edit. If I exported the raw top 200, an editor would get a list with nothing to act on. So the exported queue (section 5) ranks the **actionable** pages on their own: `review_title_meta` and `check_indexing_and_intent`. The 414 `high_volume` pages are listed as a separate count for an owner to look at.

### The decay / refresh insight

The week-4 audit tested whether pages that have gone longest without an update decline more often. Here it is on this pool, with page counts, and with a second check on the last-30-days against the previous-30-days impressions.

In [3]:
bins = [-1, 30, 90, 180, 100000]; labels = ["0-30", "31-90", "91-180", "181+"]
pool["stale_tier"] = pd.cut(pool["days_since_last_update"], bins=bins, labels=labels)
pool["declining"] = (pool["trend_direction"] == "down").astype(int)
pool["impr_change_30d"] = (pool["impressions_last_30d"] - pool["impressions_prev_30d"]) / pool["impressions_prev_30d"].replace(0, np.nan)
stale = (pool.groupby("stale_tier", observed=True)
             .agg(pages=("declining", "size"), share_declining=("declining", "mean"),
                  median_impr_change=("impr_change_30d", "median")).round(3))
print(stale.to_string())
print("\nShare of pages trending down overall:", round(pool["declining"].mean(), 3))

fig, ax = plt.subplots(figsize=(6, 3.4))
ok = stale[stale["pages"] >= 50]
ax.bar(ok.index.astype(str), ok["share_declining"], color="#1f6feb")
for i, (t, r) in enumerate(ok.iterrows()):
    ax.text(i, r["share_declining"] + .01, f"n={int(r['pages']):,}", ha="center", fontsize=8)
ax.set_ylim(0, 1); ax.set_ylabel("share of pages trending down"); ax.set_xlabel("days since last update")
ax.set_title("Staleness vs decline (tiers with 50+ pages)")
fig.tight_layout(); fig.savefig(FIG / "staleness_vs_decline.png", dpi=150); plt.close(fig)

            pages  share_declining  median_impr_change
stale_tier                                            
0-30         7613            0.587              -0.291
31-90          60            0.450              -0.154
91-180       4340            0.620              -0.302
181+           10            1.000              -0.520

Share of pages trending down overall: 0.599


**Insight (observed, this pool).** About 60% of pages in the pool are trending down. Within the pool the share moves little with staleness: 0.59 for pages updated in the last 30 days against 0.62 for the 91-180 tier. Across the whole file the rise is larger but still modest, 0.51 against 0.61. The 31-90 tier (60 pages in the pool) and the 181+ tier (10 pages) are too small to read. So older pages decline a little more, and "refresh the oldest pages first" has only weak support in this file. The playbook ranks on the CTR gap instead, and treats a refresh as one possible action a reviewer may choose, not a default.

This is a different finding from the paper's freshness result, and the two can both be true: the paper compares across a much larger inventory with a 365+ day bucket, and this pool has almost no pages that old. I only claim what this slice shows.

## 2. Intended use and limits

**Who uses it.** An SEO editor or content lead with time to look at about 20 to 50 pages a week.

**For what.** Deciding which pages to *open first* (the editor queue, `review_title_meta` and `check_indexing_and_intent`). The queue orders the reading. The editor decides whether the title, the snippet, the intent match or nothing at all needs to change.

**Where it is valid.**
- Pages with a real position (above 0 and up to 20) and at least 500 impressions in the 90-day window.
- A starter slice of 12,023 pages from 28 clients, one 90-day window.

**Where it stops.**
- No future window. The queue says a page is below its expected CTR *now*. It does not say that the gap will persist or that an edit will close it.
- The expected CTR comes from a model with a small edge over a lookup table (about 4% lower error, held-out clients). It is a reference point for a human, not a forecast.
- No query data. A low CTR may come from branded queries, a featured snippet, an AI answer or a different query mix. None of those can be seen here.
- `top_3` pages (458 in this pool) sit in a tier whose median CTR is lower than page 1's, so the expectation there is shaky.
- One client holds about 37% of the pool, so pooled numbers lean toward that site.
- Non-production. It is a worked plan, with no scheduler, no alerting and no write access to anything.

## 3. Human review + the no-go list

**Before anyone acts on a row, a person checks:**
1. Open the page's results listing. Is there a featured snippet, an image pack or an AI answer above it?
2. Is the page ranking mostly for the brand name? If yes, leave the title alone.
3. Does the title and meta description match what the main query is asking for (the `main_intent` column is a hint, not an answer)?
4. Has the page changed in the last 30 days? If so, wait a cycle before judging the CTR.
5. For `zero_clicks_visible`: is the page indexed, canonical and the version people land on? Zero clicks across 90 days can mean no result was shown.
6. Write down the change and the date, so a later comparison is possible.

**Should NOT be automated (no-go list):**
- Rewriting titles or meta descriptions without a person reading them first.
- Any change to `brand_heavy` pages without a named owner's sign-off.
- Pruning, merging or redirecting a page because it appears in the queue. The queue does not contain the information to make that call.
- Reporting a "clicks recovered" figure from the click gap. The gap is a ranking device, not a forecast.
- Applying this queue to a site that was not in the pool, or to a different time window, without rerunning the checks.
- Telling a client that an edit "caused" a change in clicks. There is no experiment behind that statement.

### Cost and value, in plain numbers

Value is bounded by the click gap. Cost is reviewer time. Neither is a forecast.

In [4]:
total_gap = queue["click_gap"].sum()
review_pool = actionable[actionable["action"] == "review_title_meta"]
print(f"Click gap summed over the pool (clicks per 90 days, upper bound, not a forecast): {total_gap:,.0f}")
print(f"Actual clicks in the pool over the same window: {int(queue['clicks_90d'].sum()):,}")
for k in (20, 50):
    t = review_pool.head(k)
    print(f"Top {k} 'review_title_meta' pages: gap {t['click_gap'].sum():,.0f} clicks/90d "
          f"= {100 * t['click_gap'].sum() / total_gap:.1f}% of the pool gap")
mins = 15
print(f"\nAt about {mins} minutes of human review per page, 50 pages is roughly {50 * mins / 60:.1f} reviewer hours a week.")
hv = queue.loc[queue["archetype"] == "high_volume", "click_gap"].sum()
print(f"{100 * hv / total_gap:.0f}% of the pool gap sits on the {int((queue['archetype'] == 'high_volume').sum())} high-volume pages, which a person has to vet before any edit.")
print("Value per hour is a ceiling set by the gap, and a page that turns out to be a branded-query page is worth nothing.")

Click gap summed over the pool (clicks per 90 days, upper bound, not a forecast): 118,452
Actual clicks in the pool over the same window: 425,112
Top 20 'review_title_meta' pages: gap 3,172 clicks/90d = 2.7% of the pool gap
Top 50 'review_title_meta' pages: gap 6,430 clicks/90d = 5.4% of the pool gap

At about 15 minutes of human review per page, 50 pages is roughly 12.5 reviewer hours a week.
39% of the pool gap sits on the 414 high-volume pages, which a person has to vet before any edit.
Value per hour is a ceiling set by the gap, and a page that turns out to be a branded-query page is worth nothing.


## 4. Monitoring / retrain triggers

Light and manual. The aim is to notice when the recommendations have gone stale, not to build an alerting system. Baselines come from this run.

In [5]:
base = json.load(open(OUT / "model_metrics.json"))
triggers = pd.DataFrame([
    ["rerun this notebook", "every new 90-day window, or monthly", "refresh the queue; nothing is live"],
    ["grouped model error rises", f"weighted error above {1.2 * base['model_mae']:.3f} (20% over today's {base['model_mae']:.3f})", "retrain, or fall back to the tier-median rule"],
    ["model stops beating the rule", f"model error not below rule error ({base['rule_mae']:.3f}) on held-out clients", "drop the model, keep the rule plus reason codes"],
    ["queue fills with dead pages", f"zero-click share of the top 50 above 10% (today {base['top50_zero_clicks_model']} of 50)", "check indexing and the pool filter"],
    ["one client dominates", f"largest client above 45% of the pool (today {100 * pool['client_id'].value_counts(normalize=True).iloc[0]:.0f}%)", "report per client, not pooled"],
    ["reviewer feedback", "more than 1 in 3 top-50 pages marked 'not a real gap' by the reviewer", "revisit the rule and the expected-CTR model"],
], columns=["watch", "threshold", "response"])
print(triggers.to_string(index=False))

                       watch                                                             threshold                                        response
         rerun this notebook                                   every new 90-day window, or monthly              refresh the queue; nothing is live
   grouped model error rises                   weighted error above 0.294 (20% over today's 0.245)   retrain, or fall back to the tier-median rule
model stops beating the rule          model error not below rule error (0.255) on held-out clients drop the model, keep the rule plus reason codes
 queue fills with dead pages              zero-click share of the top 50 above 10% (today 1 of 50)              check indexing and the pool filter
        one client dominates                      largest client above 45% of the pool (today 37%)                   report per client, not pooled
           reviewer feedback more than 1 in 3 top-50 pages marked 'not a real gap' by the reviewer     revisit the rul

## 5. Exports for the paper

Three things leave this notebook: the ranked queue (a CSV, which stays out of git by design and is rebuilt by this notebook), the figures the paper reuses (committed under `work/figures/`), and a metrics JSON with the receipts every paper number traces back to.

In [6]:
export_cols = ["rank", "content_id", "position_tier", "avg_position", "impressions_90d", "clicks_90d", "ctr",
               "expected_ctr", "click_gap", "archetype", "reason_code", "action"]
actionable.head(200)[["action_rank"] + export_cols[1:]].to_csv(OUT / "action_queue.csv", index=False)

# staleness figure: only tiers with enough pages are plotted, the others are listed in the table above
receipts = {
    "pool_pages": int(len(queue)), "clients": int(queue["client_id"].nunique()),
    "queue_exported_rows": 200, "actionable_pages": int(len(actionable)),
    "high_volume_pages": int((queue["archetype"] == "high_volume").sum()),
    "archetype_counts": {k: int(v) for k, v in queue["archetype"].value_counts().items()},
    "action_counts": {k: int(v) for k, v in queue["action"].value_counts().items()},
    "top50_raw_gap_archetype_mix": {k: int(v) for k, v in queue.head(50)["archetype"].value_counts().items()},
    "high_volume_gap_share_pct": round(100 * float(hv) / float(total_gap), 1),
    "total_click_gap": round(float(total_gap), 0), "pool_clicks_90d": int(queue["clicks_90d"].sum()),
    "top50_review_gap_share_pct": round(100 * review_pool.head(50)["click_gap"].sum() / total_gap, 1),
    "share_declining": round(float(pool["declining"].mean()), 3),
    "staleness_all_pages": {str(k): round(float(x), 3) for k, x in df.assign(d=(df["trend_direction"] == "down").astype(int)).groupby(pd.cut(df["days_since_last_update"], bins, labels=labels), observed=True)["d"].mean().items()},
    "staleness_table": {str(k): {"pages": int(v["pages"]), "share_declining": float(v["share_declining"])} for k, v in stale.iterrows()},
    "largest_client_share": round(float(pool["client_id"].value_counts(normalize=True).iloc[0]), 3),
    "triggers": triggers.to_dict(orient="records")}
json.dump(receipts, open(OUT / "playbook_metrics.json", "w"), indent=2)
order = ["over_performing", "holding", "under_clicking", "visible_ignored", "high_volume"]
cnt = queue["archetype"].value_counts().reindex(order)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.barh(order[::-1], cnt.values[::-1], color=["#d1242f", "#bf8700", "#1f6feb", "#8c959f", "#8c959f"])
for i, v in enumerate(cnt.values[::-1]): ax.text(v + 60, i, f"{v:,}", va="center", fontsize=8)
ax.set_xlabel("pages in the pool"); ax.set_title("Where the 12,023 pages land")
fig.tight_layout(); fig.savefig(FIG / "archetype_counts.png", dpi=150); plt.close(fig)
print("Wrote work/outputs/action_queue.csv (editor queue, top 200 of %d actionable pages; gitignored)" % len(actionable))
print("Wrote, work/outputs/playbook_metrics.json")
print("Figures in work/figures:", sorted(p.name for p in FIG.glob("*.png")))

Wrote work/outputs/action_queue.csv (editor queue, top 200 of 3679 actionable pages; gitignored)
Wrote, work/outputs/playbook_metrics.json
Figures in work/figures: ['archetype_counts.png', 'before_after_split.png', 'fold_errors.png', 'staleness_vs_decline.png']


## Self-check

- [x] Every section above is filled, with the thinking in markdown and the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime, then Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit the repo URL on the card.